# Visualise Learned fMRI Embeddings

This notebook projects encoder embeddings into 2-D using **t-SNE** and **UMAP**,
colouring points by:
- Pre vs post surgery condition
- Functional network
- Subject identity


In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.manifold import TSNE
import umap

from model.encoder    import Encoder
from training.dataset import SubjectICADataset
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
ENCODER_WEIGHTS = '../outputs/best_encoder.pth'   # ← update path
CSV_PATH        = '../data/your_data.csv'          # ← update path
TEMPORAL_DIM    = 590
BATCH_SIZE      = 8

In [ ]:
# ── Load encoder ───────────────────────────────────────────────────────────
encoder = Encoder(temporal_input_dim=TEMPORAL_DIM).to(device)
encoder.load_state_dict(torch.load(ENCODER_WEIGHTS, map_location=device))
encoder.eval()
print('Encoder loaded.')

In [ ]:
# ── Build dataset and dataloader ───────────────────────────────────────────
df      = pd.read_csv(CSV_PATH)
dataset = SubjectICADataset(df, temporal_resample=TEMPORAL_DIM)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f'{len(dataset)} subject-network pairs loaded.')

In [ ]:
# ── Extract embeddings ─────────────────────────────────────────────────────
all_emb, all_cond, all_net, all_subj = [], [], [], []

with torch.no_grad():
    for batch in loader:
        for view, cond in [('pre', 'pre'), ('post', 'post')]:
            emb = encoder(
                batch[f'{view}_spatial'].to(device),
                batch[f'{view}_temporal'].to(device)
            ).cpu().numpy()
            all_emb.append(emb)
            all_cond.extend([cond]          * len(emb))
            all_net.extend(batch['network'])
            all_subj.extend(batch['subject'])

embeddings = np.concatenate(all_emb, axis=0)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# ── t-SNE projection ───────────────────────────────────────────────────────
perplexity = min(30, len(embeddings) // 4)
tsne_2d = TSNE(n_components=2, perplexity=perplexity, random_state=42).fit_transform(embeddings)
print('t-SNE done.')

In [ ]:
# ── Plot: condition (pre vs post) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colours = {'pre': '#4C72B0', 'post': '#DD8452'}
for cond in ['pre', 'post']:
    mask = [c == cond for c in all_cond]
    ax.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1],
               c=colours[cond], label=cond, alpha=0.7, s=20)
ax.set_title('t-SNE: Pre vs Post Surgery')
ax.legend()
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('tsne_condition.png', dpi=150)
plt.show()

In [ ]:
# ── Plot: functional network ───────────────────────────────────────────────
unique_nets = sorted(set(all_net))
palette     = cm.tab10(np.linspace(0, 1, len(unique_nets)))

fig, ax = plt.subplots(figsize=(9, 6))
for net, col in zip(unique_nets, palette):
    mask = [n == net for n in all_net]
    ax.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1],
               c=[col], label=net, alpha=0.7, s=20)
ax.set_title('t-SNE: Functional Networks')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('tsne_network.png', dpi=150)
plt.show()

In [ ]:
# ── UMAP projection ────────────────────────────────────────────────────────
reducer   = umap.UMAP(n_components=2, random_state=42)
umap_2d   = reducer.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, coords, title in zip(axes, [tsne_2d, umap_2d], ['t-SNE', 'UMAP']):
    for cond, col in colours.items():
        mask = [c == cond for c in all_cond]
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=col, label=cond, alpha=0.7, s=15)
    ax.set_title(f'{title}: Pre vs Post')
    ax.legend()
plt.tight_layout()
plt.savefig('umap_vs_tsne.png', dpi=150)
plt.show()